# Intent Routing and Classification

The chapter opener asked what a search system should *do* with "Bücher von Goethe".
By now the pipeline can see the query clearly: the tokenizer splits it, the language
detector recognizes German, POS tagging marks "Bücher" as a plural noun and "Goethe"
as a proper noun, and named-entity recognition promotes "Goethe" to a person. What
has not been decided is where the query should go: a book catalogue, a web index, an
author database, or a general-purpose model. This demo turns the extracted structure
into that routing decision.

Two of the pipeline stages are the same classification problem underneath. Language
detection sorts a query into one of the languages the system knows; intent routing
sorts it into one of the backends the system supports. Both are text classification,
and both fall to the same Naive Bayes model. We build that model once, from scratch,
then point it at both problems.

**Learning goals:**
- Read the Naive Bayes decision rule and implement it in log-space with Laplace smoothing
- Detect language with a from-scratch classifier over character n-grams, where Chapter 3.1's rules gave up
- Route queries to backends using POS, NER, and question-form features
- See why intent routing wants a uniform prior, not the empirical one
- Trace one query end-to-end through every stage of the chapter

**Prerequisites:** Sections 3.1 to 3.4 (the full text-processing pipeline)

In [1]:
import math
import re
import warnings
from collections import Counter, defaultdict

import nltk
import spacy
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

from shared.display import print_table, display_md
from shared.text import stopwords_for

In [2]:
# One-time downloads and model loading (quiet, safe to re-run)
warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)

NLP = {"en": spacy.load("en_core_web_sm"), "de": spacy.load("de_core_news_sm")}
STEM = {"en": SnowballStemmer("english").stem, "de": SnowballStemmer("german").stem}
SW = {"en": stopwords_for("en"), "de": stopwords_for("de")}

display_md(
    "Loaded spaCy models `en_core_web_sm` and `de_core_news_sm`, Snowball stemmers, "
    "and NLTK stop-word lists. Everything below runs offline and is deterministic: "
    "Naive Bayes has no random component, so re-running reproduces every number."
)

Loaded spaCy models `en_core_web_sm` and `de_core_news_sm`, Snowball stemmers, and NLTK stop-word lists. Everything below runs offline and is deterministic: Naive Bayes has no random component, so re-running reproduces every number.

## 1. The end-to-end query pipeline

A query enters as a raw string and flows through the classical stages this chapter
built. The order is not arbitrary: language detection must precede any
language-specific step, POS tagging must precede selective stop-word filtering, and
NER must precede intent classification because entity types are input features to the
classifier.

The stages run in this order:

| # | Pipeline stage | Source |
| ---: | --- | --- |
| 1 | Tokenize and normalize | Section 3.1 |
| 2 | Detect language | Section 3.1 rules, plus this demo's classifier |
| 3 | Stop-word filter | Section 3.2 (language-specific) |
| 4 | Stem or lemmatize | Section 3.2 (language-specific) |
| 5 | POS-tag and extract entities | Section 3.4 |
| 6 | Classify intent | This demo |
| 7 | Route to backend | This demo |

What comes out is a structured representation a downstream system can act on: a
language, a set of entities, an intent label, and a cleaned token list. The last two
stages, language detection and intent routing, are the classification steps. We turn
to the model that powers both.

## 2. Naive Bayes for text classification

Naive Bayes picks the most probable class given the observed features. It applies
Bayes' theorem and assumes the features are conditionally independent given the class.
For a feature vector $\mathbf{x} = (x_1, \ldots, x_M)$ and classes $C_1, \ldots, C_K$:

$$\hat{C} = \arg\max_{k} \; P(C_k) \cdot \prod_{j=1}^{M} P(x_j \mid C_k)$$

Implementations never compute that product directly. Multiplying hundreds of small
probabilities underflows to zero in floating point, so we take logs and the product
becomes a sum:

$$\hat{C} = \arg\max_{k} \left( \log P(C_k) + \sum_{j} c_j \log P(x_j \mid C_k) \right)$$

where $c_j$ is the count of feature $x_j$ in the input. Three quantities are learned
from a labelled corpus:

- **The prior** $P(C_k)$: the fraction of training documents in class $C_k$. When the
  classes are balanced or the class balance is unknown, a uniform prior $1/K$ is used
  and the term drops out of the argmax.
- **The likelihood** $P(x_j \mid C_k)$: the fraction of feature occurrences in class
  $C_k$ that land on feature $x_j$.
- **A smoothing constant**. If $x_j$ never appeared in the training data for class
  $C_k$, its maximum-likelihood probability is zero, and a single zero collapses the
  whole product. Add-one (Laplace) smoothing adds a small constant to every count so
  no feature is ever impossible.

**Key insight:** the product is a direct consequence of the independence assumption.
Real text has tens of thousands of feature dimensions, so a full vector $\mathbf{x}$
almost never recurs and its joint probability cannot be estimated. Each single
dimension $x_j$ is seen many times, so its marginal $P(x_j \mid C_k)$ can be counted.
Independence trades a joint probability we cannot measure for a product of marginals
we can. The assumption is false (tokens are not independent), yet the errors tend to
hit every class similarly and rarely flip the argmax, which is why Naive Bayes stays
in production stacks two decades after neural classifiers arrived.

In [3]:
class NaiveBayes:
    """Multinomial Naive Bayes over feature lists, from scratch, in log-space."""

    def __init__(self, alpha=1.0, uniform_prior=False):
        self.alpha = alpha                       # Laplace smoothing constant
        self.uniform_prior = uniform_prior       # ignore class base rates if True
        self.class_counts = Counter()            # class -> number of training docs
        self.feature_counts = defaultdict(Counter)  # class -> feature -> count
        self.total_features = Counter()          # class -> total feature occurrences
        self.vocab = set()                       # all features seen in training

    def train(self, documents, labels):
        for features, label in zip(documents, labels):
            self.class_counts[label] += 1
            for f in features:                   # a repeated feature counts twice
                self.feature_counts[label][f] += 1
                self.total_features[label] += 1
                self.vocab.add(f)

    def log_prior(self, label):
        if self.uniform_prior:
            return math.log(1.0 / len(self.class_counts))
        return math.log(self.class_counts[label] / sum(self.class_counts.values()))

    def log_likelihood(self, feature, label):
        # (count + alpha) / (class total + alpha * vocab size): add-one smoothing
        count = self.feature_counts[label][feature]
        total = self.total_features[label]
        return math.log((count + self.alpha) / (total + self.alpha * len(self.vocab)))

    def scores(self, features):
        return {
            label: self.log_prior(label) + sum(self.log_likelihood(f, label) for f in features)
            for label in self.class_counts
        }

    def predict(self, features):
        s = self.scores(features)
        return s, max(s, key=s.get)

    def posterior(self, features):
        """Normalized posterior probabilities (softmax over log-scores)."""
        s = self.scores(features)
        top = max(s.values())
        exp = {k: math.exp(v - top) for k, v in s.items()}
        z = sum(exp.values())
        return {k: v / z for k, v in exp.items()}

**Why log-space?** Twenty features each with probability 0.01:

- Direct product: $0.01^{20}$ = 1.0e-40, which underflows toward zero.
- Log-space sum: $20 \cdot \log(0.01)$ = -92.1, a comfortable number.

**Why Laplace smoothing?** If a feature never occurred in class $C$ during training,
its unsmoothed likelihood is 0, and one zero in the product drives $P(C \mid \mathbf{x})$
to 0 no matter how well every other feature fits. Add-one smoothing pretends every
feature in the vocabulary was seen once more than it was, so an unseen feature is merely
unlikely, not impossible.

## 3. Language detection as classification

Language detection uses character n-grams as features. Each language has short
character sequences that rarely appear in others: `sch`, `cht`, `und` mark German;
`the`, `ing`, `ion` mark English; `les`, `des`, `que` mark French. We pad every word
with `_` so that word-initial and word-final patterns become features in their own
right (the Cavnar-Trenkle convention), because languages differ in how words begin
and end, not only in their interior.

In [4]:
def char_ngrams(text, n):
    """Character n-grams per word, each word padded with '_' at both boundaries."""
    grams = []
    for word in re.findall(r"\w+", text.lower()):
        padded = "_" + word + "_"
        grams += [padded[i:i + n] for i in range(len(padded) - n + 1)]
    return grams

def lang_features(text):
    """Bigram and trigram profile of a text."""
    return char_ngrams(text, 2) + char_ngrams(text, 3)

# The trigrams of the running query, exactly as the book lists them
display_md(
    "**Trigrams of `Bücher von Goethe`** (n=3, boundary-padded):\n\n"
    f"`{char_ngrams('Bücher von Goethe', 3)}`\n\n"
    "The word-initial `_bü`, `büc`, `üch` carry the `ü` diacritic that only German uses "
    "here; `_vo`, `von`, `on_` are the common German preposition; `_go`, `goe`, `oet` "
    "match German name patterns. Almost every trigram points one way."
)

**Trigrams of `Bücher von Goethe`** (n=3, boundary-padded):

`['_bü', 'büc', 'üch', 'che', 'her', 'er_', '_vo', 'von', 'on_', '_go', 'goe', 'oet', 'eth', 'the', 'he_']`

The word-initial `_bü`, `büc`, `üch` carry the `ü` diacritic that only German uses here; `_vo`, `von`, `on_` are the common German preposition; `_go`, `goe`, `oet` match German name patterns. Almost every trigram points one way.

We train on a dozen short, everyday sentences per language. The sentences are ordinary
prose, not tuned to any test query, but they naturally contain the frequent function
words (`der`, `die`, `von` for German; `le`, `les`, `de` for French) and diacritics
that carry most of the discriminating signal.

In [5]:
TRAIN_EN = [
    "The weather is nice today", "I would like to buy a book",
    "She went to the store yesterday", "The university offers many courses",
    "Please send me the information", "How much does this cost",
    "The meeting starts at three", "I really enjoyed the lecture",
    "Can you help me find the library", "The train arrives at noon",
    "Children are playing in the garden", "This is the best restaurant in town",
    "My computer is not working", "The books of the author are famous",
    "Where is the nearest station", "The house on the hill is old",
    "We had a great time on the weekend", "The president gave a long speech",
]
TRAIN_DE = [
    "Das Wetter ist heute schön", "Ich möchte ein Buch kaufen",
    "Sie ging gestern in den Laden", "Die Universität bietet viele Kurse an",
    "Bitte schicken Sie mir die Informationen", "Wie viel kostet das",
    "Die Besprechung beginnt um drei", "Die Vorlesung hat mir gut gefallen",
    "Können Sie mir die Bibliothek zeigen", "Der Zug kommt um Mittag an",
    "Die Kinder spielen im Garten", "Das ist das beste Restaurant der Stadt",
    "Mein Computer funktioniert nicht", "Die Bücher von dem Autor sind berühmt",
    "Wo ist der nächste Bahnhof", "Das Haus auf dem Hügel ist alt",
    "Wir hatten am Wochenende eine schöne Zeit", "Der Präsident hielt eine lange Rede",
]
TRAIN_FR = [
    "Le temps est beau aujourd'hui", "Je voudrais acheter un livre",
    "Elle est allée au magasin hier", "L'université offre beaucoup de cours",
    "Envoyez-moi les informations s'il vous plaît", "Combien cela coûte-t-il",
    "La réunion commence à trois heures", "J'ai vraiment apprécié le cours",
    "Pouvez-vous m'aider à trouver la bibliothèque", "Le train arrive à midi",
    "Les enfants jouent dans le jardin", "Le meilleur restaurant de la ville",
    "Mon ordinateur ne fonctionne pas", "Les livres de cet auteur sont célèbres",
    "Où est la gare la plus proche", "La maison sur la colline est vieille",
    "Nous avons passé un bon week-end", "Le président a fait un long discours",
]

lang_corpus = ([(s, "en") for s in TRAIN_EN]
               + [(s, "de") for s in TRAIN_DE]
               + [(s, "fr") for s in TRAIN_FR])

lang_nb = NaiveBayes(alpha=0.5)
lang_nb.train([lang_features(s) for s, _ in lang_corpus],
              [lang for _, lang in lang_corpus])

display_md(
    f"**Language detector trained.** {sum(lang_nb.class_counts.values())} sentences, "
    f"{dict(lang_nb.class_counts)} per class, "
    f"{len(lang_nb.vocab):,} distinct n-gram features (the profile). "
    "A production detector like `lingua` keeps hundreds of thousands of n-grams per "
    "language; this toy profile is enough to separate three languages."
)

**Language detector trained.** 54 sentences, {'en': 18, 'de': 18, 'fr': 18} per class, 1,007 distinct n-gram features (the profile). A production detector like `lingua` keeps hundreds of thousands of n-grams per language; this toy profile is enough to separate three languages.

In [6]:
# The most frequent n-grams per language: the profile the classifier leans on
rows = []
for lang in ("en", "de", "fr"):
    top = [g for g, _ in lang_nb.feature_counts[lang].most_common(12)]
    rows.append([lang, ", ".join(top)])
print_table(rows, headers=["Language", "Most frequent n-grams (boundary '_' marks a word edge)"])

| Language   | Most frequent n-grams (boundary '_' marks a word edge)   |
|:-----------|:---------------------------------------------------------|
| en         | e_, _t, th, he, _th, the, he_, s_, re, n_, _i, t_        |
| de         | e_, _d, t_, n_, ie, en, r_, te, er, de, ch, en_          |
| fr         | e_, s_, _l, le, t_, ou, es, _a, n_, ur, on, _le          |

Now classify full sentences the model never saw. On sentence-length input the
evidence piles up and the winner is unambiguous.

In [7]:
LANG_NAMES = {"en": "English", "de": "German", "fr": "French"}

def detect_language(text):
    """Return (language, confidence, full posterior) from the trained detector."""
    post = lang_nb.posterior(lang_features(text))
    best = max(post, key=post.get)
    return best, post[best], post

sentence_tests = [
    ("Where is the train station?", "en"),
    ("Wo ist der Bahnhof?", "de"),
    ("Où est la gare?", "fr"),
    ("The book is on the table", "en"),
    ("Das Buch liegt auf dem Tisch", "de"),
    ("Le livre est sur la table", "fr"),
]
rows = []
for text, gold in sentence_tests:
    lang, conf, _ = detect_language(text)
    rows.append([text, LANG_NAMES[gold], LANG_NAMES[lang], f"{conf:.2f}",
                 "correct" if lang == gold else "WRONG"])
print_table(rows, headers=["Sentence", "Gold", "Detected", "Confidence", "Result"])

| Sentence                     | Gold    | Detected   |   Confidence | Result   |
|:-----------------------------|:--------|:-----------|-------------:|:---------|
| Where is the train station?  | English | English    |            1 | correct  |
| Wo ist der Bahnhof?          | German  | German     |            1 | correct  |
| Où est la gare?              | French  | French     |            1 | correct  |
| The book is on the table     | English | English    |            1 | correct  |
| Das Buch liegt auf dem Tisch | German  | German     |            1 | correct  |
| Le livre est sur la table    | French  | French     |            1 | correct  |

Section 3.1's rule-based detector could not decide short queries with no stop word and
no diacritic: it returned "undetermined" for "Mein computer" and "pain". The
statistical detector has more to go on, because even short strings carry n-grams.

In [8]:
short_tests = ["Bücher von Goethe", "Mein computer", "pain", "die", "test", "film"]
rows = []
for q in short_tests:
    lang, conf, post = detect_language(q)
    ordered = sorted(post.items(), key=lambda kv: kv[1], reverse=True)
    runner = LANG_NAMES[ordered[1][0]]
    rows.append([q, LANG_NAMES[lang], f"{conf:.2f}", runner])
print_table(rows, headers=["Short query", "Detected", "Confidence", "Runner-up"])

| Short query       | Detected   |   Confidence | Runner-up   |
|:------------------|:-----------|-------------:|:------------|
| Bücher von Goethe | German     |         0.97 | English     |
| Mein computer     | German     |         1    | English     |
| pain              | French     |         0.99 | English     |
| die               | German     |         1    | French      |
| test              | French     |         0.54 | English     |
| film              | English    |         0.67 | French      |

The classifier decides every one of these, where the rules abstained:

- **`Bücher von Goethe`** lands on German at 0.97, essentially the 0.96 the production library `lingua` reports. The `ü`, `von`, and name trigrams above win it.
- **`Mein computer`** is German: `mein` and the word-shape agree, even though `computer` looks international.
- **`die`** is a false friend: an English verb and the German article. The n-gram shape is decisively German, so the detector is confident and, for an English sentence using the verb, would be confidently wrong. Short input removes the context that would break the tie.
- **`test`** and **`film`** score almost evenly across languages (low confidence): the tokens genuinely look plausible in more than one language. A real system falls back to a default or the user's locale when the top confidence is this close to the runner-up.

**Caution:** confidence on very short input is unreliable. One or two discriminating
n-grams can push the posterior near 1.0 even when a human would call the query
ambiguous, and a false friend like "die" is decided with false certainty. Route on the
top language only when its confidence clears a margin over the runner-up; otherwise
fall back to a default.

Because the classifier is written from scratch, it is worth checking it against a
library implementation. scikit-learn's `MultinomialNB` on the same features and
smoothing should agree on every prediction.

In [9]:
texts = [s for s, _ in lang_corpus]
labels = [lang for _, lang in lang_corpus]
vectorizer = CountVectorizer(analyzer=lang_features)
matrix = vectorizer.fit_transform(texts)
sk_nb = MultinomialNB(alpha=0.5)
sk_nb.fit(matrix, labels)

check = [t for t, _ in sentence_tests] + short_tests
sk_pred = sk_nb.predict(vectorizer.transform(check))
rows = [[q, LANG_NAMES[detect_language(q)[0]], LANG_NAMES[p]]
        for q, p in zip(check, sk_pred)]
agree = all(detect_language(q)[0] == p for q, p in zip(check, sk_pred))
print_table(rows, headers=["Query", "From scratch", "scikit-learn"])
display_md(f"**All predictions agree: {agree}.** The from-scratch model is the real thing, "
           "not an approximation of it.")

| Query                        | From scratch   | scikit-learn   |
|:-----------------------------|:---------------|:---------------|
| Where is the train station?  | English        | English        |
| Wo ist der Bahnhof?          | German         | German         |
| Où est la gare?              | French         | French         |
| The book is on the table     | English        | English        |
| Das Buch liegt auf dem Tisch | German         | German         |
| Le livre est sur la table    | French         | French         |
| Bücher von Goethe            | German         | German         |
| Mein computer                | German         | German         |
| pain                         | French         | French         |
| die                          | German         | German         |
| test                         | French         | French         |
| film                         | English        | English        |

**All predictions agree: True.** The from-scratch model is the real thing, not an approximation of it.

## 4. Intent routing as classification

The same model routes the query to a backend. Once the language is known, an intent
classifier decides which of the system's supported backends the query wants. We use a
representative subset of the book's intents:

- `book_search`: titles, authors, ISBNs
- `web_search`: general keyword search
- `people_search`: information about a person
- `news_search`: current events, temporal subjects
- `map_search`: locations and directions
- `product_search`: shoppable products

Features are richer than for language detection because more of the pipeline's output
is available. For each query we combine, exactly as the book proposes:

- the content tokens after stop-word removal and stemming (a bag of words)
- question-form markers: does it start with a WH-word, does it end with `?`
- NER labels present: has-PERSON, has-LOCATION, has-DATE, has-MONEY
- the detected language, as a categorical feature

In [10]:
WH_WORDS = {"who", "what", "when", "where", "which", "why", "how", "whose", "whom",
            "wer", "was", "wann", "wo", "welche", "warum", "wie"}

def intent_features(query, lang):
    """POS/NER/question-form features for a query in a known language."""
    doc = NLP[lang](query)
    tokens = [t.text.lower() for t in doc]
    feats = ["LANG_" + lang]
    for t in doc:                                    # content-word bag of stems
        w = t.text.lower()
        if w.isalpha() and w not in SW[lang] and len(w) > 1:
            feats.append("w=" + STEM[lang](w))
    if tokens and tokens[0] in WH_WORDS:
        feats.append("WH_FIRST")
    if any(t in WH_WORDS for t in tokens):
        feats.append("HAS_WH")
    if "?" in query:
        feats.append("QMARK")
    ent_labels = {e.label_ for e in doc.ents}
    if ent_labels & {"PERSON", "PER"}:
        feats.append("HAS_PERSON")
    if ent_labels & {"GPE", "LOC"}:
        feats.append("HAS_LOCATION")
    if ent_labels & {"DATE", "TIME"}:
        feats.append("HAS_DATE")
    if ent_labels & {"MONEY"}:
        feats.append("HAS_MONEY")
    return feats

# The features for one query, so the abstraction is not a black box
display_md(
    "**Features for `Who is Albert Einstein?`** (English):\n\n"
    f"`{intent_features('Who is Albert Einstein?', 'en')}`\n\n"
    "The stemmed content words plus a WH-word at the front, a question mark, and a "
    "PERSON entity: the combination a `people_search` query typically shows."
)

**Features for `Who is Albert Einstein?`** (English):

`['LANG_en', 'w=albert', 'w=einstein', 'WH_FIRST', 'HAS_WH', 'QMARK', 'HAS_PERSON']`

The stemmed content words plus a WH-word at the front, a question mark, and a PERSON entity: the combination a `people_search` query typically shows.

The training data is a small set of labelled queries, mostly English with a few German
ones so the router is multilingual. Each query is featurized in its own language.

In [11]:
intent_corpus = [
    ("books by Jane Austen", "en", "book_search"),
    ("novels written by Goethe", "en", "book_search"),
    ("best science fiction books", "en", "book_search"),
    ("the hobbit hardcover edition", "en", "book_search"),
    ("author of the great gatsby", "en", "book_search"),
    ("recommended books about history", "en", "book_search"),
    ("Romane von Friedrich Schiller", "de", "book_search"),
    ("Bücher über die Geschichte Europas", "de", "book_search"),
    ("how do neural networks work", "en", "web_search"),
    ("what is information retrieval", "en", "web_search"),
    ("tips for better sleep", "en", "web_search"),
    ("difference between tcp and udp", "en", "web_search"),
    ("why is the sky blue", "en", "web_search"),
    ("recipe for apple pie", "en", "web_search"),
    ("who is Albert Einstein", "en", "people_search"),
    ("biography of Marie Curie", "en", "people_search"),
    ("when was Barack Obama born", "en", "people_search"),
    ("what did Isaac Newton discover", "en", "people_search"),
    ("age of Roger Federer", "en", "people_search"),
    ("where does Elon Musk live", "en", "people_search"),
    ("wer ist Angela Merkel", "de", "people_search"),
    ("latest news about the election", "en", "news_search"),
    ("who won the game yesterday", "en", "news_search"),
    ("stock market today", "en", "news_search"),
    ("earthquake in Japan this morning", "en", "news_search"),
    ("covid cases this week", "en", "news_search"),
    ("what happened at the summit today", "en", "news_search"),
    ("directions from Basel to Zurich", "en", "map_search"),
    ("restaurants near me", "en", "map_search"),
    ("how to get to the airport", "en", "map_search"),
    ("map of central London", "en", "map_search"),
    ("distance between Paris and Berlin", "en", "map_search"),
    ("coffee shops in Manhattan", "en", "map_search"),
    ("Weg von Basel nach Zürich", "de", "map_search"),
    ("order running shoes online", "en", "product_search"),
    ("cheap laptop under 500 dollars", "en", "product_search"),
    ("best price for iphone 15", "en", "product_search"),
    ("discount code for nike sneakers", "en", "product_search"),
    ("where to buy a coffee machine", "en", "product_search"),
    ("wireless keyboard for sale", "en", "product_search"),
]

intent_nb = NaiveBayes(alpha=1.0, uniform_prior=True)
intent_nb.train([intent_features(q, lang) for q, lang, _ in intent_corpus],
                [label for _, _, label in intent_corpus])

display_md(
    f"**Intent classifier trained** on {len(intent_corpus)} queries, "
    f"{len(intent_nb.class_counts)} intents, {len(intent_nb.vocab)} features. "
    "Trained with a **uniform prior** on purpose: see the caution below."
)

**Intent classifier trained** on 40 queries, 6 intents, 117 features. Trained with a **uniform prior** on purpose: see the caution below.

**Caution: intent routing wants a uniform prior.** Query-log training data reflects
the *current* backend distribution, not the ideal one. If 80% of past queries went to
web search, an empirical prior routes new queries to web search 80% of the time
regardless of their features, because the prior term overwhelms weak feature evidence.
Use a uniform prior and let the features drive the decision, or balance the training
data across intents before estimating priors.

Now classify new queries, each first passed through the language detector from
Section 3 so the correct spaCy model is used.

In [12]:
intent_tests = [
    "Who is Albert Einstein?",
    "who won the F1 race on the weekend",
    "books written by Hemingway",
    "order running shoes online",
    "how does a search engine work",
    "directions to the train station",
    "best price for a used car",
]
rows = []
for q in intent_tests:
    lang = detect_language(q)[0]
    lang = lang if lang in NLP else "en"          # only en/de models loaded
    scores, pred = intent_nb.predict(intent_features(q, lang))
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    gap = ordered[0][1] - ordered[1][1]
    rows.append([q, pred, f"{gap:.1f}", ordered[1][0]])
print_table(rows, headers=["Query", "Predicted intent", "Log-score gap", "Runner-up"])

| Query                              | Predicted intent   |   Log-score gap | Runner-up     |
|:-----------------------------------|:-------------------|----------------:|:--------------|
| Who is Albert Einstein?            | people_search      |             4   | web_search    |
| who won the F1 race on the weekend | news_search        |             1.5 | map_search    |
| books written by Hemingway         | book_search        |             1.4 | people_search |
| order running shoes online         | product_search     |             2.7 | web_search    |
| how does a search engine work      | web_search         |             0.5 | people_search |
| directions to the train station    | map_search         |             0.6 | web_search    |
| best price for a used car          | product_search     |             0.9 | book_search   |

The gap is the margin of the winning log-score over the runner-up: a large gap means a
confident routing decision, a small gap a close call worth escalating. To see the
argmax at work, here are all six class log-scores for one query.

In [13]:
q = "Who is Albert Einstein?"
scores, pred = intent_nb.predict(intent_features(q, "en"))
rows = [[intent, f"{score:.1f}", "  <- winner" if intent == pred else ""]
        for intent, score in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]
print_table(rows, headers=["Intent", "Log-score", ""])
display_md(
    f"**`{q}`** routes to **{pred}**. The WH-word, question mark, and PERSON entity all "
    "push toward `people_search`; no other class collects that combination of features."
)

| Intent         |   Log-score |           |
|:---------------|------------:|:----------|
| people_search  |       -28.4 | <- winner |
| web_search     |       -31.9 |           |
| product_search |       -32.7 |           |
| news_search    |       -32.7 |           |
| map_search     |       -33.5 |           |
| book_search    |       -34   |           |

**`Who is Albert Einstein?`** routes to **people_search**. The WH-word, question mark, and PERSON entity all push toward `people_search`; no other class collects that combination of features.

## 5. End-to-end walkthrough

Putting all five sections together on the running query. Each stage consumes the
previous stage's output and adds one layer of structure, ending in a backend and a
structured query.

In [14]:
def run_pipeline(query):
    """Trace a query through every stage of the chapter."""
    stages = []

    tokens = re.findall(r"\w+", query)
    stages.append(("Tokenize", str(tokens)))

    normalized = [t.lower() for t in tokens]
    stages.append(("Normalize (case)", str(normalized)))

    lang, conf, _ = detect_language(query)
    lang = lang if lang in NLP else "en"
    stages.append(("Detect language", f"{LANG_NAMES.get(lang, lang)} (confidence {conf:.2f})"))

    content = [t for t in normalized if t not in SW[lang]]
    dropped = [t for t in normalized if t in SW[lang]]
    stages.append(("Stop-word filter", f"drop {dropped}  ->  {content}"))

    stemmed = [STEM[lang](t) for t in content]
    stages.append(("Stem", str(stemmed)))

    doc = NLP[lang](query)
    pos = [(t.text, t.pos_) for t in doc]
    stages.append(("POS tag", str(pos)))

    ents = [(e.text, e.label_) for e in doc.ents]
    stages.append(("Named entities", str(ents) if ents else "(none)"))

    _, intent = intent_nb.predict(intent_features(query, lang))
    stages.append(("Classify intent", intent))

    return stages, lang, ents, intent

stages, lang, ents, intent = run_pipeline("Bücher von Goethe")
print_table([[name, out] for name, out in stages], headers=["Stage", "Output"])

| Stage            | Output                                                    |
|:-----------------|:----------------------------------------------------------|
| Tokenize         | ['Bücher', 'von', 'Goethe']                               |
| Normalize (case) | ['bücher', 'von', 'goethe']                               |
| Detect language  | German (confidence 0.97)                                  |
| Stop-word filter | drop ['von']  ->  ['bücher', 'goethe']                    |
| Stem             | ['buch', 'goeth']                                         |
| POS tag          | [('Bücher', 'NOUN'), ('von', 'ADP'), ('Goethe', 'PROPN')] |
| Named entities   | [('Goethe', 'PER')]                                       |
| Classify intent  | book_search                                               |

In [15]:
author = next((text for text, label in ents if label in {"PERSON", "PER"}), None)
display_md(
    "**Route.** The pipeline turned three tokens into a structured backend query:\n\n"
    f"| Field | Value |\n|---|---|\n"
    f"| backend | `{intent}` |\n"
    f"| language | `{lang}` |\n"
    f"| author | `{author}` |\n"
    f"| content | `*` |\n\n"
    "The library catalogue answers with every Goethe title it holds. No general-purpose "
    "keyword match against the raw string `Bücher von Goethe` was ever needed: the "
    "language feature and the PERSON entity, not the literal words, drove the decision."
)

**Route.** The pipeline turned three tokens into a structured backend query:

| Field | Value |
|---|---|
| backend | `book_search` |
| language | `de` |
| author | `Goethe` |
| content | `*` |

The library catalogue answers with every Goethe title it holds. No general-purpose keyword match against the raw string `Bücher von Goethe` was ever needed: the language feature and the PERSON entity, not the literal words, drove the decision.

The same machinery handles the other chapter-opener query, "Who won the F1 race on the
weekend?". It detects English, tags "who" as a WH-word, recognizes a DATE entity ("the
weekend"), and routes to `news_search` on the strength of the question form plus the
temporal entity. The classical pipeline scopes the query; the semantic-search and
retrieval-augmented-generation chapters carry it the rest of the way to a direct
answer.

In [16]:
stages, lang, ents, intent = run_pipeline("who won the F1 race on the weekend")
print_table([[name, out] for name, out in stages], headers=["Stage", "Output"])

| Stage            | Output                                                                                                                                    |
|:-----------------|:------------------------------------------------------------------------------------------------------------------------------------------|
| Tokenize         | ['who', 'won', 'the', 'F1', 'race', 'on', 'the', 'weekend']                                                                               |
| Normalize (case) | ['who', 'won', 'the', 'f1', 'race', 'on', 'the', 'weekend']                                                                               |
| Detect language  | English (confidence 1.00)                                                                                                                 |
| Stop-word filter | drop ['who', 'won', 'the', 'on', 'the']  ->  ['f1', 'race', 'weekend']                                                                    |
| Stem             | ['f1', 'race', 'weekend']                                                                                                                 |
| POS tag          | [('who', 'PRON'), ('won', 'VERB'), ('the', 'DET'), ('F1', 'PROPN'), ('race', 'NOUN'), ('on', 'ADP'), ('the', 'DET'), ('weekend', 'NOUN')] |
| Named entities   | [('F1', 'GPE'), ('the weekend', 'DATE')]                                                                                                  |
| Classify intent  | news_search                                                                                                                               |

## Summary

| Component | Mechanism | Why it matters |
| --- | --- | --- |
| Naive Bayes | prior x product of per-feature likelihoods, in log-space | Fast, interpretable, trains on few examples |
| Laplace smoothing | add-one to every count | Stops one unseen feature from zeroing a class |
| Language detection | NB over character n-grams (n=2,3) | Decides short queries the rules could not |
| Intent routing | NB over token, POS, NER, and language features | Dispatches queries to specialized backends |
| Uniform prior | ignore class base rates | Keeps a skewed query log from dominating routing |

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Takeaway</strong><br>
Language detection and intent routing are the same classification problem with different features and classes, and one from-scratch Naive Bayes model solves both. Character n-grams decide language where Chapter 3.1's rules abstained; POS, NER, and question-form features turn "one search box, one algorithm" into "many backends, intelligently dispatched". Production stacks now favour gradient-boosted trees or small transformers because feature interactions matter, and open-ended assistants let a language model do the whole pipeline in one prompt. Naive Bayes remains the pedagogical baseline and the cost-sensitive first pass, and its calibrated posteriors are exactly what a system needs to decide when to escalate.
</div>

## Try it yourself

1. Add five more training sentences per language in Section 3. Does the confidence on
   "test" and "film" rise or does the ambiguity persist?
2. Feed an English sentence that uses the verb "die" ("plants die without water") to
   `detect_language`. Does sentence context now break the false-friend tie that the
   bare word "die" could not?
3. Add a seventh intent (for example `weather`) with a handful of training queries.
   Does the classifier keep the other six intents correct?
4. Run `run_pipeline` on a query of your own and read the stages. Where would a wrong
   language detection have derailed everything downstream?

In [17]:
my_query = "Bücher über Physik"
stages, lang, ents, intent = run_pipeline(my_query)
display_md(
    f"**Your query:** `{my_query}`\n\n"
    f"- Detected language: **{LANG_NAMES.get(lang, lang)}**\n"
    f"- Entities: `{ents}`\n"
    f"- Routed to: **{intent}**"
)

**Your query:** `Bücher über Physik`

- Detected language: **German**
- Entities: `[]`
- Routed to: **book_search**